## Load new data and assess gene overlap with DepMap

New gene dependency data isn't guaranteed to cover the same genes, use the
same gene identifiers, or list them in the same order as the DepMap data
BioBombe was trained on.
This notebook loads the new dataset and reports exactly how it lines up
against DepMap and against the trained ensemble's gene set, before anything
downstream trusts a join between them.

Run this script with `8.apply-biobombe-new-data/` as the working directory.

In [1]:
import pathlib
import sys

import pandas as pd

sys.path.insert(0, "utils")
import data_prep as dp

In [2]:
data_directory = pathlib.Path("../0.data-download/data").resolve()
model_save_dir = pathlib.Path("../3.run-biobombe/saved_models").resolve()

# Point this at the new dataset once it's available; see the module README.
NF1_data_path = pathlib.Path("data/largaespada/NF1_data.parquet")

# Everything derived from the new dataset itself, not just DepMap's public
# data, stays under data/ (gitignored) rather than the committed results/.
results_dir = pathlib.Path("data/largaespada/results")
results_dir.mkdir(parents=True, exist_ok=True)

In [3]:
depmap_df, gene_dict_df = dp.load_depmap_reference(data_directory)
trained_gene_order = dp.get_trained_gene_order(model_save_dir)

print(f"DepMap reference: {depmap_df.shape[0]} models, {depmap_df.shape[1] - 1} genes")
print(f"Trained ensemble gene set: {len(trained_gene_order)} genes")

DepMap reference: 1150 models, 18443 genes
Trained ensemble gene set: 2718 genes


In [4]:
new_df = dp.load_new_dependency_data(NF1_data_path)
print(f"New data: {new_df.shape[0]} samples, {new_df.shape[1] - 1} genes")
new_df.head()

New data: 8 samples, 19168 genes


,ModelID,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
0,S462,-0.076632,-0.123356,0.000292,-0.078742,0.019515,0.157764,0.131372,0.305262,0.213588,...,-0.220339,-0.086933,-0.073876,-0.244316,-0.276291,-0.166674,-0.148169,-0.316441,-0.502940,-0.554147
1,ST88-14,0.115021,0.065473,-0.006382,0.045658,0.246857,-0.022087,0.020388,-0.452509,0.172521,...,-0.136272,-0.051020,0.368793,0.214840,0.257873,0.085956,-0.098698,0.124072,-0.077770,0.022499
2,STS-26T,0.034313,0.043876,-0.048730,0.008211,-0.009792,0.040197,0.260923,-0.479493,-0.032089,...,-0.106695,-1.462055,-0.170383,0.035239,0.116699,0.260284,-0.139800,0.204675,0.149275,0.117490
3,iHSC1L_C3,0.080343,0.044938,-0.067431,0.041823,0.121697,-0.061105,0.075610,-0.432215,0.149937,...,-0.166183,-0.474575,0.025829,0.012389,0.011572,-0.006098,-0.102745,0.120065,-0.087963,-0.308699
4,iHSC1L_F12,0.059759,0.343791,-0.096634,0.107508,-0.023710,-0.069656,0.085512,-0.423115,0.061635,...,-0.163819,-0.501199,0.114948,0.018613,0.064420,0.037258,-0.143688,0.067039,-0.143123,-0.328702


In [5]:
overlap_report = dp.assess_gene_overlap(new_df, depmap_df, gene_dict_df, trained_gene_order)

print(f"Resolved {overlap_report['n_resolved']} / {overlap_report['n_new_genes']} new-data gene columns to an entrez id")
print(f"Shared with DepMap's full gene set: {overlap_report['n_shared_with_depmap']} / {len(overlap_report['depmap_entrez_ids'])}")
print(f"Shared with the trained ensemble's gene set: {overlap_report['n_shared_with_trained']} / {len(overlap_report['trained_entrez_ids'])}")
print(f"Trained genes with no match in the new data: {len(overlap_report['trained_genes_missing_in_new'])}")
print(f"New data already in DepMap gene order: {overlap_report['is_order_aligned']}")

if overlap_report["unresolved_columns"]:
    print(f"\n{len(overlap_report['unresolved_columns'])} new-data columns couldn't be resolved to an entrez id, "
          "first 10:")
    print(overlap_report["unresolved_columns"][:10])

if overlap_report["trained_genes_missing_in_new"]:
    coverage = 1 - len(overlap_report["trained_genes_missing_in_new"]) / len(trained_gene_order)
    print(f"\nTrained-gene coverage: {coverage:.1%}. "
          "Missing genes get imputed with the training mean in step 2, not dropped.")

Resolved 18062 / 19168 new-data gene columns to an entrez id
Shared with DepMap's full gene set: 18062 / 18443
Shared with the trained ensemble's gene set: 2701 / 2718
Trained genes with no match in the new data: 17
New data already in DepMap gene order: True

1106 new-data columns couldn't be resolved to an entrez id, first 10:
['ABCF2', 'ABO', 'ACKR5', 'ADGB', 'ADGRF2P', 'ADISSP', 'AFG2A', 'AFG2B', 'AGAP11', 'AGAP4']

Trained-gene coverage: 99.4%. Missing genes get imputed with the training mean in step 2, not dropped.


In [6]:
# Save the full report for the later steps and for manual review.
# entrez_map and the two gene-name lists are the parts worth keeping on disk;
# the entrez id sets are large and easily recomputed, so they're left out.
report_to_save = pd.DataFrame({
    "NF1_data_column": list(overlap_report["entrez_map"].keys()),
    "resolved_entrez_id": list(overlap_report["entrez_map"].values()),
})
report_to_save.to_parquet(results_dir / "gene_overlap_report.parquet", index=False)

missing_genes_df = pd.DataFrame({"trained_gene_missing_in_new": overlap_report["trained_genes_missing_in_new"]})
missing_genes_df.to_parquet(results_dir / "trained_genes_missing_in_new.parquet", index=False)

print(f"Saved gene overlap report to {results_dir / 'gene_overlap_report.parquet'}")

Saved gene overlap report to data/largaespada/results/gene_overlap_report.parquet
